In [1]:
import pandas as pd
import geopandas as gpd

def load_collision_data(path: str) -> gpd.GeoDataFrame:
    df = pd.read_csv(path, sep=";", encoding="latin1")

    df = df.rename(columns={
        "Datum": "datetime",
        "Viltslag": "species",
        "Län": "lan",
        "Kommun": "kommun",
        "Lat WGS84": "lat",
        "Long WGS84": "lon",
    })

    # Fix Swedish decimal commas
    df["lat"] = df["lat"].astype(str).str.replace(",", ".", regex=False)
    df["lon"] = df["lon"].astype(str).str.replace(",", ".", regex=False)

    # Convert types
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", dayfirst=False)

    # Drop bad rows
    df = df.dropna(subset=["datetime", "lat", "lon"]).copy()

    # Build GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    # Reproject to Sweden metric CRS
    gdf = gdf.to_crs("EPSG:3006")

    return gdf

In [2]:
import pandas as pd
from grid import create_grid
from grid import spatial_join_points_to_grid

def build_cell_month_table_full(gdf_points, cell_size=5000):
    """
    Build a full cell-month panel:
    one row per grid cell per month, including zero-collision months.
    """

    # 1. Create grid
    grid = create_grid(gdf_points, cell_size=cell_size)

    # 2. Join points to grid
    joined = spatial_join_points_to_grid(gdf_points, grid)
    joined = joined.dropna(subset=["cell_id"]).copy()
    joined["cell_id"] = joined["cell_id"].astype(int)

    # 3. Month column
    joined["period_start"] = joined["datetime"].dt.to_period("M").dt.to_timestamp()

    # 4. Count collisions per observed cell-month
    observed = (
        joined.groupby(["cell_id", "period_start"])
        .size()
        .reset_index(name="collision_count")
    )

    # 5. Build full set of months in study period
    min_month = gdf_points["datetime"].dt.to_period("M").min().to_timestamp()
    max_month = gdf_points["datetime"].dt.to_period("M").max().to_timestamp()

    all_months = pd.date_range(start=min_month, end=max_month, freq="MS")

    # 6. Build full cell x month panel
    full_index = pd.MultiIndex.from_product(
        [grid["cell_id"].astype(int).unique(), all_months],
        names=["cell_id", "period_start"]
    )

    full_panel = full_index.to_frame(index=False)

    # 7. Merge observed counts into full panel
    cell_month = full_panel.merge(
        observed,
        on=["cell_id", "period_start"],
        how="left"
    )

    cell_month["collision_count"] = cell_month["collision_count"].fillna(0).astype(int)

    # 8. Define risk
    threshold = cell_month["collision_count"].median()
    cell_month["risk"] = (cell_month["collision_count"] > threshold).astype(int)

    return grid, joined, cell_month

In [3]:
gdf = load_collision_data("C:/Users/amand/PycharmProjects/thesis_boogaloo/data/Collisions/Rådata 2025.csv")

grid, joined, cell_month = build_cell_month_table_full(gdf, cell_size=5000)

print(cell_month.head())
print(cell_month.shape)
print(cell_month["collision_count"].describe())
print(cell_month["risk"].value_counts())

   cell_id period_start  collision_count  risk
0        0   2026-01-01                0     0
1        0   2026-02-01                0     0
2        0   2026-03-01                0     0
3        0   2026-04-01                0     0
4        1   2026-01-01                0     0
(2556312, 4)
count    2.556312e+06
mean     7.810471e-03
std      1.571115e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.200000e+01
Name: collision_count, dtype: float64
risk
0    2546012
1      10300
Name: count, dtype: int64


In [4]:
from roads import build_road_features, load_roads_for_study_area, load_roads

roads = load_roads_for_study_area(
    path="C:/Users/amand/PycharmProjects/thesis_boogaloo/data/Sverige_Vägtrafiknät_GeoPackage/Sverige_Vägtrafiknät_194602.gpkg",
    gdf_points=gdf,
    buffer_m=2000
)

road_features = build_road_features(
    grid=grid,
    roads=roads,
    keep_only_classes=["bilnät"]
)

model_df = cell_month.merge(
    road_features.drop(columns="geometry"),
    on="cell_id",
    how="left"
)

model_df = model_df[model_df["road_length_m"] > 0].copy()

print(model_df.shape)
print(model_df["risk"].value_counts())
print(model_df[["road_length_m", "road_density", "nearest_road_distance_m"]].describe())

C:\Users\amand\PycharmProjects\thesis_boogaloo\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 669 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
C:\Users\amand\PycharmProjects\thesis_boogaloo\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 669 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


(65704, 9)
risk
0    55353
1    10351
Name: count, dtype: int64
       road_length_m  road_density  nearest_road_distance_m
count   65704.000000  6.570400e+04             65704.000000
mean    39737.808415  1.589512e-03               435.260344
std     28949.770073  1.157991e-03               580.560670
min         0.001104  4.415033e-11                 0.007941
25%     21313.649766  8.525460e-04                95.693230
50%     34539.178864  1.381567e-03               227.788486
75%     53604.944920  2.144198e-03               490.406819
max    397091.751566  1.588367e-02              3518.845814


In [5]:
relevant_cell_ids = model_df["cell_id"].unique()
grid_small = grid[grid["cell_id"].isin(relevant_cell_ids)].copy()

print("Original grid:", grid.shape)
print("Filtered grid:", grid_small.shape)

Original grid: (639078, 2)
Filtered grid: (16343, 2)


In [6]:
from weather import build_cell_month_temperature

temperature_features = build_cell_month_temperature(grid=grid_small)

model_df = model_df.merge(
    temperature_features[["cell_id", "period_start", "temp_mean", "temp_min", "temp_max"]],
    on=["cell_id", "period_start"],
    how="left"
)

model_df = model_df.dropna(subset=["temp_mean", "temp_min", "temp_max"]).copy()

print(model_df[["temp_mean", "temp_min", "temp_max"]].isna().mean())

C:\Users\amand\PycharmProjects\thesis_boogaloo\src\weather.py:131: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids["geometry"] = centroids.geometry.centroid


Parsed columns for station 72500: ['Datum', 'Tid (UTC)', 'Lufttemperatur', 'Kvalitet', 'Unnamed: 4', 'Tidsutsnitt:']
date_col: Datum
time_col: Tid (UTC)
temp_col: Lufttemperatur
Metadata request failed for station 11032: 404
Parsed columns for station 84140: ['Datum', 'Tid (UTC)', 'Lufttemperatur', 'Kvalitet', 'Unnamed: 4', 'Tidsutsnitt:']
date_col: Datum
time_col: Tid (UTC)
temp_col: Lufttemperatur
Parsed columns for station 125490: ['Datum', 'Tid (UTC)', 'Lufttemperatur', 'Kvalitet', 'Unnamed: 4', 'Tidsutsnitt:']
date_col: Datum
time_col: Tid (UTC)
temp_col: Lufttemperatur
Parsed columns for station 146050: ['Datum', 'Tid (UTC)', 'Lufttemperatur', 'Kvalitet', 'Unnamed: 4', 'Tidsutsnitt:']
date_col: Datum
time_col: Tid (UTC)
temp_col: Lufttemperatur
Parsed columns for station 156770: ['Datum', 'Tid (UTC)', 'Lufttemperatur', 'Kvalitet', 'Unnamed: 4', 'Tidsutsnitt:']
date_col: Datum
time_col: Tid (UTC)
temp_col: Lufttemperatur
Parsed columns for station 65020: ['Datum', 'Tid (UTC)', 'Lu

In [7]:
from weather import fetch_temperature_stations_from_api, assign_nearest_temperature_station
stations = fetch_temperature_stations_from_api()
print(stations.shape)
print(stations.head())

cell_station = assign_nearest_temperature_station(grid, stations)
print(cell_station.head())
print(cell_station.shape)
print(cell_station["station_id"].nunique())

(997, 7)
  station_id     station_name      lat      lon   height country  active
0     154860  Abelvattnet Aut  65.5300  14.9700  665.000    None   False
1     188800           Abisko  68.3538  18.8166  393.380    None   False
2     188790       Abisko Aut  68.3538  18.8164  392.235    None    True
3     158990           Abraur  65.9857  18.9195  368.079    None    True
4      85600         Adelsnäs  58.1998  15.9802   97.000    None   False


C:\Users\amand\PycharmProjects\thesis_boogaloo\src\weather.py:131: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids["geometry"] = centroids.geometry.centroid


   cell_id station_id station_name  station_distance_km
0      0.0      52240  Falsterbo A          6264.551983
1      1.0      52240  Falsterbo A          6259.749974
2      2.0      52240  Falsterbo A          6254.948172
3      3.0      52240  Falsterbo A          6250.146576
4      4.0      52240  Falsterbo A          6245.345188
(639078, 4)
986


In [11]:
features = [
    "road_length_m",
    "road_density",
    "nearest_road_distance_m",
    "temp_mean",
    "temp_min",
    "temp_max",
]

model_df_clean = model_df.dropna(subset=features).copy()

print("After dropna:", model_df_clean.shape)
print(model_df_clean["period_start"].value_counts().sort_index())

After dropna: (4335, 12)
period_start
2026-01-01    4335
Name: count, dtype: int64


In [9]:
months = sorted(model_df["period_start"].unique())
print(months)

train_months = months[:-1]   # all except last
test_months = months[-1:]    # last month

train = model_df[model_df["period_start"].isin(train_months)].copy()
test = model_df[model_df["period_start"].isin(test_months)].copy()

X_train = train[features]
y_train = train["risk"]

X_test = test[features]
y_test = test["risk"]

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())


[Timestamp('2026-01-01 00:00:00')]
train shape: (0, 6)
test shape: (4335, 6)
Series([], Name: count, dtype: int64)
risk
0    3524
1     811
Name: count, dtype: int64


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000, class_weight="balanced")

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

ValueError: Found array with 0 sample(s) (shape=(0, 6)) while a minimum of 1 is required by LogisticRegression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))